<a href="https://colab.research.google.com/github/eulaia/pibic-audiodescricao-llms/blob/main/Gpt4o_audiodescricao_imagens.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# INSTALAÇÃO
# ============================================================

!pip install openai -q

# ============================================================
# IMPORTAÇÕES
# ============================================================

import os
import base64

from datetime import datetime
from typing import List
from openai import AsyncOpenAI
from google.colab import drive
from IPython.display import display
from PIL import Image

# ============================================================
# GOOGLE DRIVE
# ============================================================

drive.mount("/content/drive")

# ============================================================
# CONFIGURAÇÕES
# ============================================================

OPENAI_API_KEY = ""

IMAGE_DIRECTORY = "/content/drive/MyDrive/imagens"

MODEL_NAME = "gpt-4o"

MAX_RESPONSE_TOKENS = 300

SUPPORTED_IMAGE_EXTENSIONS = (
    ".png",
    ".jpg",
    ".jpeg"
)

# ============================================================
# CLIENTE OPENAI
# ============================================================

client = AsyncOpenAI(
    api_key=OPENAI_API_KEY
)

# ============================================================
# PROMPT
# ============================================================

PROMPT = """
Descreva a imagem de maneira objetiva e concisa, com foco em facilitar a interpretação
por um leitor de audiodescrição. Inclua os seguintes elementos:
Foco Principal: Identifique o principal sujeito ou objeto da imagem. Perspectiva da Foto:
Descreva a perspectiva da foto (ex: de frente, de lado, de cima, etc.) e o tipo de plano (ex: plano
geral, plano médio, close-up, etc.). Enquadramento: Explique como os elementos principais
estão posicionados e enquadrados na imagem. Plano de Fundo: Descreva o que está no fundo da
imagem e como ele contribui para a cena. Iluminação: Explique a iluminação da cena e como ela
afeta a percepção dos elementos na imagem. Contexto (se fornecido): Inclua detalhes sobre o
local e a época, se disponíveis. Preciso que a resposta seja no formato de parágrafo completo e
coerente, não faça em partes.
"""

# ============================================================
# UTILITÁRIOS
# ============================================================

def get_image_files(
    directory_path: str
) -> List[str]:

    if not os.path.exists(directory_path):
        raise FileNotFoundError(
            f"Pasta não encontrada: {directory_path}"
        )

    return [
        file_name
        for file_name in os.listdir(directory_path)
        if file_name.lower().endswith(
            SUPPORTED_IMAGE_EXTENSIONS
        )
    ]


def convert_image_to_base64(
    image_path: str
) -> str:

    with open(image_path, "rb") as image_file:
        return base64.b64encode(
            image_file.read()
        ).decode("utf-8")


def get_image_mime_type(
    image_name: str
) -> str:

    if image_name.lower().endswith(".png"):
        return "image/png"

    return "image/jpeg"

# ============================================================
# CONTEÚDO MULTIMODAL
# ============================================================

def build_multimodal_request(
    image_base64: str,
    mime_type: str
) -> List[dict]:

    return [
        {
            "type": "text",
            "text": PROMPT
        },
        {
            "type": "image_url",
            "image_url": {
                "url": (
                    f"data:{mime_type};base64,"
                    f"{image_base64}"
                )
            }
        }
    ]

# ============================================================
# GERAÇÃO DA DESCRIÇÃO
# ============================================================

async def generate_image_description(
    image_base64: str,
    mime_type: str
) -> str:

    multimodal_content = build_multimodal_request(
        image_base64,
        mime_type
    )

    response = await client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": multimodal_content
            }
        ],
        max_tokens=MAX_RESPONSE_TOKENS
    )

    generated_description = (
        response.choices[0]
        .message
        .content
    )

    return generated_description

# ============================================================
# PROCESSAMENTO INDIVIDUAL
# ============================================================

async def process_image(
    image_name: str
):

    image_path = os.path.join(
        IMAGE_DIRECTORY,
        image_name
    )

    print("\n" + "=" * 80)
    print(f"Processando imagem: {image_name}")

    image_base64 = convert_image_to_base64(
        image_path
    )

    mime_type = get_image_mime_type(
        image_name
    )

    print("\nGerando descrição...\n")

    description = await generate_image_description(
        image_base64,
        mime_type
    )

    image = Image.open(image_path)
    display(image)

    print("\nDESCRIÇÃO:\n")
    print(description)

    print("\n" + "=" * 80)

# ============================================================
# PROCESSAMENTO GERAL
# ============================================================

async def process_all_images():

    current_datetime = datetime.now()

    print("=" * 80)
    print("GERAÇÃO DE AUDIODESCRIÇÃO DE IMAGENS")
    print("=" * 80)

    image_files = get_image_files(
        IMAGE_DIRECTORY
    )

    print(
        f"Data: "
        f"{current_datetime.strftime('%d/%m/%Y')}"
    )

    print(
        f"Horário: "
        f"{current_datetime.strftime('%H:%M:%S')}"
    )

    print(f"Modelo utilizado: {MODEL_NAME}")
    print(f"Quantidade de imagens: {len(image_files)}")

    if len(image_files) == 0:
        print("\nNenhuma imagem encontrada.")
        return

    for image_name in image_files:

        try:
            await process_image(
                image_name
            )

        except Exception as error:
            print(
                f"\nErro ao processar "
                f"{image_name}: {error}"
            )

    print("\nProcessamento finalizado.")

# ============================================================
# EXECUÇÃO
# ============================================================

await process_all_images()